# Initialize

## Load Libraries

In [ ]:
import matplotlib
import random
import numpy as np
import matplotlib.pyplot as plt

## Load Data

In [ ]:
data  = np.transpose(np.loadtxt('g-Factor.txt', skiprows=1, delimiter = '\t'))

# This is to skip the last few points of the data set
padding = -100
t = data[0,:padding]
gx = data[1,:padding]
gy = data[2,:padding]
gz = data[3,:padding]

In [ ]:
fs = 12
fs_x = 14
fs_y = 9

# Expected value for g
Eidgenössisches Institut für Metrologie METAS:
https://www.metas.ch/metas/de/home/dok/gravitationszonen.html

In [ ]:
g_theo = 9.806 # N/kg

# Main

In [ ]:
# Get size of data set
n = gz.shape[0]

print('Number of data points : {:d}'.format(n))

dt = (t[-1] - t[0]) / n 

print('Time per data point : {:0.2f} ms'.format(dt * 1000))

## Plot Data

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(t, gz, 'b.', label='Data')

# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

## Discretize data by sorting it into bins

In [ ]:
# Number of bins and bin size

gz_max = np.max(gz)
gz_min = np.min(gz)


bin_size = 0.01
#bin_size = 0.005
#bin_size = 0.002

n_bins = int((gz_max - gz_min) / bin_size) + 1

bins = np.linspace(gz_min, gz_min+bin_size*(n_bins-1), n_bins)

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(t, gz, 'b.', label='Data', zorder = 1)
for b in bins:
    ax1.hlines(b, 0, t[-1], 'r', zorder = 2)

# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

In [ ]:
# Initialize array holding number of data  points (occurrence) per bin
occurrence = np.zeros(n_bins)

# Calculate the offset of the occurrence-array index with respect to the floor division of the data
offset = np.min(bins)//bin_size + 1

# Iterate through all data and find the index of the occurrence-array into which the data falls.
# Then increment this array value by one. 
for gz_i in gz:
    index = int(gz_i//bin_size-offset)
    occurrence[index] = occurrence[index] + 1

## Plot Histogram

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence), 'r', label='Theoretical value')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Occurrences')
ax1.legend()
plt.show()

## Calculate Mean and Variance

In [ ]:
mean = np.sum(gz)/n
var = np.sum((gz-mean)**2)/(n-1)

In [ ]:
print('Mean : {:0.2f} N/kg, Standard deviation (std) : {:0.2f} N/kg'.format(mean, np.sqrt(var)))

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence), 'r', label='Theoretical value')
ax1.vlines(mean, 0, np.max(occurrence), 'g', label='Mean')
ax1.vlines(mean + np.sqrt(var), 0, np.max(occurrence), 'g', linestyles='dashed', label='Mean +/- std')
ax1.vlines(mean - np.sqrt(var), 0, np.max(occurrence), 'g', linestyles='dashed')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Occurrences')
ax1.legend()
plt.show()

<b>Question:</b> Are the data points normally distributed?

In [ ]:
# Calculate the fraction of the data point within a certain interval around the mean.

# Sum of the data points in the interval
n_sigma = 0
# Width of the interval in terms of the standard deviation
width = 1
# iterate through all bins and sum their values if they lie in the specified interval
i = 0
for b in bins:
    if (b >= (mean - np.sqrt(var) * width)) and (b <= (mean + np.sqrt(var) * width)):
        n_sigma = n_sigma + occurrence[i]
    i = i + 1
    
print('Occurences within the {:d}-sigma-interval: {:0.1f}%'.format(width,n_sigma/n*100))

## Probability Distribution

In [ ]:
# Probability normalization factor (the favorable over the possible)
A = n

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence/A, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence/A), 'r')
ax1.vlines(mean, 0, np.max(occurrence/A), 'g')
ax1.vlines(mean + np.sqrt(var), 0, np.max(occurrence/A), 'g', linestyles='dashed')
ax1.vlines(mean - np.sqrt(var), 0, np.max(occurrence/A), 'g', linestyles='dashed')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Probability')
plt.show()

## Deviation from Theory Value

In [ ]:
d = (g_theo - mean) / np.sqrt(var)

In [ ]:
print('Deviation of the mean from the theory value: {:0.1f} sigma'.format(d))